# Stock Price Agent 
A rule-based multi-stock monitoring agent built using Python and yfinance.

## Features:
- Multi-stock tracking
- OHLC price analysis
- Trend-aware decisions
- Profit/Loss tracking
- P/E ratio & dividend insights
- Alert system
- Finite execution loop
  

## Relevant Definitions
- A **Dividend** is a portion of a company’s profits paid to shareholders.
- **Dividend Yield** shows how much income a stock generates relative to its price.

- **Price formatting** refers to displaying stock prices in a clean, readable format.
  
- **PnL** stands for *Profit and Loss*.  
It measures how much money you have gained or lost on an investment.

- The **P/E ratio** measures how expensive a stock is relative to its earnings.

- **Trailing P/E** is a type of P/E ratio based on **past earnings (last 12 months)**.



In [1]:
# -----------------------------
# Imports
# -----------------------------
import yfinance as yf 
import time 

In [2]:
# -----------------------------
# Fetch Stock Data (OHLC + Prev Close)
# -----------------------------
# provides Open, High, Low, Close (OHLC)
# previous market close --> allows for detection of trend 
def get_stock_data(ticker):
    stock = yf.Ticker(ticker)
    
    hist = stock.history(period="2d")  # get 2 days for comparison
    
    if hist.empty:
        return None
        
    latest = hist.iloc[-1]

    data = {
        "close":latest["Close"],
        "open":latest["Open"],
        "high": latest["High"],
        "low": latest["Low"],
    }    
    # Previous close (for trend)
    if len(hist) > 1:
        data["prev_close"] = hist.iloc[-2]["Close"]
    else:
        data["prev_close"] = None 
        
    return data

In [3]:
# -----------------------------
# Smarter Decision Logic
# -----------------------------
# Aware of trends 
def smarter_decision(data, threshold):
    if data is None:
        return "No data"
        
    price = data[ "close"]
    prev_price = data["prev_close"]

    if prev_price is None:
        return "No trend data"

    # Trend 
    trend = "Up" if price > prev_price else "Down"

    # Decision logic 
    if price > threshold and trend == "Down":
        return "Sell (Overbought + falling)"
    elif price < threshold and trend == "Up":
        return "Buy (Recovery)"
    else:
        return "Hold"
    

In [4]:
# -----------------------------
# Alert System
# -----------------------------
def alert_user(message):
    print("Alert:", message)
    

In [5]:
# -----------------------------
# Price Formatting
# -----------------------------
def format_price(price):
    if price is None:
        return "N/A"
    return f"${price:.2f}"

In [6]:
# -----------------------------
# Portfolio + PnL
# -----------------------------
portfolio = {
    "AAPL": { "buy price": 240, "quantity" :1},
    "TSLA": {"buy_price": 200, "quantity" : 1}
}

def calculate_pnl(ticker, current_price):
    if ticker not in portfolio:
        return None 

    buy_price = portfolio[ticker]["buy price"]
    qty = portfolio[ticker]["quantity"]

    pnl = (current_price - buy_price) *qty
    return pnl

In [7]:
# -----------------------------
# Fundamentals (P/E + Dividend)
# -----------------------------
def get_fundamentals(ticker):
     stock = yf.Ticker(ticker)
     info = stock.info

     return{
         "pe_ratio":info.get("trailingPE"),
         "dividend_yield":info.get("dividendYield")
     }
     

In [8]:
# -----------------------------
# Agent Loop (Finite Execution)
# -----------------------------
def stock_agent(tickers, threshold, interval=60, iterations=3):
    print("\nStarting multi-stock agent...\n")

    for _ in range(iterations):
        for ticker in tickers:

            data = get_stock_data(ticker)

            if data is None:
                print(f"{ticker}: No data")
                continue

            decision = smarter_decision(data, threshold)
            price = data["close"]
            formatted_price = format_price(price)

            print(f"\nStock: {ticker}")
            print(f"Price: {formatted_price}")
            print(f"Open: {data['open']}, High: {data['high']}, Low: {data['low']}")
            print(f"Decision: {decision}")
            
            #PnL
            pnl = calculate_pnl(ticker, price)
            if pnl is not None:
                print(f"P/L: ${pnl:.2f}")
                
            # Fundamentals
            fundamentals = get_fundamentals(ticker)
            print(f"P/E Ratio: {fundamentals['pe_ratio']}")
            print(f"Dividend Yield: {fundamentals['dividend_yield']}")

            #alert
            if "Buy" in decision or "Sell" in decision:
                alert_user(f"{ticker}: {decision} at {formatted_price}")

        
        print("\n--- Waiting for next cycle ---\n")
        time.sleep(interval)

In [9]:
# -----------------------------
# Run Agent (User Input)
# -----------------------------
user_input = input("Enter stock tickers: ")

tickers = [t.strip().upper() for t in user_input.split(",") if t.strip()]

if not tickers:
    print("No valid tickers entered. Exiting.")
else:
    threshold = float(input("Enter threshold price: "))
    iterations = int(input("Enter number of cycles: "))
    
    stock_agent(tickers, threshold=threshold, interval=60, iterations=iterations)

Enter stock tickers:  GOOG,MSFT,NVDA,AAPL
Enter threshold price:  100
Enter number of cycles:  2



Starting multi-stock agent...


Stock: GOOG
Price: $280.30
Open: 277.8399963378906, High: 280.5400085449219, Low: 276.80999755859375
Decision: Hold
P/E Ratio: 25.906193
Dividend Yield: 0.31

Stock: MSFT
Price: $366.93
Open: 364.6300048828125, High: 368.1499938964844, Low: 364.375
Decision: Hold
P/E Ratio: 22.976204
Dividend Yield: 1.01

Stock: NVDA
Price: $170.17
Open: 166.97000122070312, High: 170.2100067138672, Low: 166.9600067138672
Decision: Hold
P/E Ratio: 34.65784
Dividend Yield: 0.02

Stock: AAPL
Price: $248.54
Open: 247.88999938964844, High: 249.27000427246094, Low: 247.63999938964844
Decision: Hold
P/L: $8.54
P/E Ratio: 31.500633
Dividend Yield: 0.42

--- Waiting for next cycle ---


Stock: GOOG
Price: $280.49
Open: 277.8399963378906, High: 280.5899963378906, Low: 276.80999755859375
Decision: Hold
P/E Ratio: 25.92329
Dividend Yield: 0.31

Stock: MSFT
Price: $366.58
Open: 364.6300048828125, High: 368.1499938964844, Low: 364.375
Decision: Hold
P/E Ratio: 22.954914
Dividend Yiel